### 第一章  什么是RAG
1. RAG（Retrieval Augmented Generation）：检索 增强 生成
2. 核心组件：LLM  检索器 向量知识库
3. 工作流程：
![RAG流程图](./img/RAG工作流程.png)
    1. 用户输入查询-prompt
    2. 检索器根据查询从知识库中检索相关文档-relevent documents
    3. prompt + relevent documents ----> Augmented prompt(增强提示词)
    4. Augmented prompt 输入到LLM中，生成最终的响应-response 
4. 优势：
    1. 提高响应质量：RAG可以根据检索到的相关文档，生成更准确、更相关的响应。
    2. 减少 hallucination（幻觉）：通过引入外部知识库，RAG可以减少模型生成的幻觉现象。
    3. 提高时效性：更新知识库信息而非重新训练模型，降低成本。
5. token表
    一个汉字 往往 对应 1个token 
    一个英文单词 往往 对应 1-3个token (unhappy ----> un happy)
    **回答相同问题时，中文回复往往消耗更少的token**(_成本优化_)

     

### 第二章  检索器
1. 检索方式
    1. 关键词检索-Keyword Search
    2. 语义检索-Semantic Search
    3. 混合检索-Hybrid Search    关键词检索 + 语义检索 + 重排序

    _元数据检索-Metadata Search    非检索技术，作为一种过滤手段，搭配上述检索技术使用_

2. 关键词检索
    1. 基于TF-IDF的关键词搜索
    2. 基于BM25的关键词搜索

3. 混合检索

    通过关键词检索和语义检索获得两个结果集，然后通过RRF重排序，取前k个文档作为最终结果集

4. 检索器评估

    1. 精确率(Precision)
    $$
    \mathrm{Precision}=\frac{检索到的相关文档数}{检索到的总文档数}
    $$
    反映检索器返回无关文档的情况,用于衡量结果的可信度

    2. 召回率(Recall)
    $$
    \mathrm{Recall}=\frac{检索到的相关文档数}{知识库中总相关文档数}
    $$
    反映检索器是否遗漏了相关文档,用于衡量结果的覆盖程度

    精确率与召回率受检索器返回文档数$k$影响,$k$通常取$5-15$    

    3. 平均精确率(Mean Average Precision)

    MAP@K平均精确率（主流评价标准）：前k个文档中搜索到了n个相关文档，那么每个文档处的精确率是n/k, 将正确文档处的精确率相加，除以搜索到的正确文档数即得到平均精确率

    ![MAP@K](./img/MAP@k.png)

    4. 平均倒数排名 MRR(Mean Reciprocal Rank)
    $$
    \mathrm{Reciprocal Rank}=\frac{1}{\mathrm{Rank}}
    $$

    衡量第一个相关文档在返回结果中的排名位置

    在4次检索中, 每次返回的文档中第一个相关文档的排名分别为$1,2,3,4$,则平均倒数排名为$\frac{1}{1}+\frac{1}{2}+\frac{1}{3}+\frac{1}{4}=1.25$。




   

### BM25算法(Best Matching 25)
最佳匹配算法在已有知识库D时,对于给定查询Q(由$q_1$、$q_2$、$q_3$等词组成),计算得分公式:$Score(D,Q)$
$$
Score(D,Q) = \sum_{t\in Q}{IDF}(t)\cdot \frac{(k_1+1) TF}{TF+k_1\left(1-b+b\frac{|d|}{\mathrm{avgdl}}\right)}
$$

**TF为词频,表示词 $t$ 在文档 $d$ 中出现的次数**

**$|d|$为文档长度**

**$k_1$为词频饱和度，可调超参数；越小，饱和速度越快，越大，曲线越接近线性增长典型取值：1.2-2.0**

**$b$为长度归一化强度，可调超参数；等于0时长度不影响权重（长文档无惩罚），等于1时完全根据文档相对长度进行归一化。经典取值：0.75**

其中IDF常用形式为：
$$
IDF(t) = \log \frac{N-n_t+0.5}{n_t+0.5}
$$
**$n_t$为包含词$t$的文档数**

**$N$为总文档数**

**$0.5$为平滑参数，用于处理空文档和空词的情况。**

$IDF(t)$反应了查询词的"稀有性",如果一个词如"的"、"好"在所有文档中都有出现,则取对数后IDF分值会极低或者为负数,若一个词如"量子代数计算"仅在极少数的文件中出现,那么取对数后IDF分值会极高,+0.5防止分母为0出现极端化的情况
  

_可以把 BM25 想象成一个打分公式，它在问：_

_这个词稀有吗？（IDF 部分）——越稀有，分越高。_

_文档里这个词出现多少次？（TF 部分）——出现越多，分越高，但是差不多得了，出现太多次（堆砌关键词）加分不明显。_

_这篇文章是不是太长（灌水）了？（长度归一化部分）——如果文章很长，但只提了几次关键词，说明主题不聚焦，要扣分；如果文章短小精悍，但提到了关键词，说明主题很聚焦，要加分。_





### RRF(Reciprocal Rank Fusion)倒数排名融合/逆序秩融合

1. RRF是一种把“多个检索/排序结果列表”融合成一个总排序的方法

2. 核心思想：
    不直接融合分数(因为不同检索器的分数不可比),而是融合"名次"
    越靠前的结果贡献越大(用名次的倒数表示)

3. 计算公式：
    $$
    \mathrm{RRFScore}(d∈D) = \sum_{r∈R}\frac{1}{k+rank_r(d)}
    $$

D:所有被检索到的文档集合。

R:所有不同的排序系统(或搜索方法)的集合(例如:一个系统是关键词搜索,另一个是语义搜索)

$rank_r(d)$:文档d在排序系统r中的排序位置(排名序号,通常)从1开始

k:一个平滑参数,通常设置为一个较小的常数(经典论文中常设为60)

### 检索率与召回率
假设知识库中有10篇文档与prompt相关,检索器返回的12个检索结果中,有8个是相关的文档,则:
$$
Precision=\frac{8}{12}=0.67
$$
$$
Recall=\frac{8}{10}=0.8
$$

若令检索返回15个检索结果,其中9个是相关的文档,则:
$$
Precision=\frac{9}{15}=0.6    (↓)
$$

$$
Recall=\frac{9}{10}=0.9    (↑)
$$


### 第三章 检索器在生产环境应用

1. 向量检索的原理
k-近邻搜索(KNN)(k-nearest neighbors search)
